# Repository note

This is an output-pruned archival copy of an early executed Account-A setup/demo notebook. Code and execution metadata are retained; outputs were removed to keep the archive compact. Original executed Kaggle SHA256: `9e8024f84f41add26eafc396990e125d823565e2cbaea37c1edfb068beb01357`.


In [1]:
# Cell 1 — Check GPU
!nvidia-smi

In [2]:
# Cell 2 — Clone repo
!cd /kaggle/working/FreeFine && git rev-parse HEAD > /kaggle/working/freefine_commit.txt
!cd /kaggle/working/FreeFine && git checkout <that-hash>
%cd /kaggle/working
!git clone https://github.com/CIawevy/FreeFine.git
%cd FreeFine

In [3]:
# Cell 3 — Install Miniforge + create Python 3.10 env
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
!bash /tmp/mf.sh -b -p /kaggle/working/miniforge
!/kaggle/working/miniforge/bin/conda create -n FreeFine python=3.10.13 -y
!/kaggle/working/miniforge/bin/conda install -n FreeFine pip -y
!ls /kaggle/working/miniforge/envs/FreeFine/bin/pip

In [4]:
# Cell 4 — Install repo requirements (exact pinned versions)
!/kaggle/working/miniforge/envs/FreeFine/bin/pip install -r /kaggle/working/FreeFine/requirements.txt
!/kaggle/working/miniforge/envs/FreeFine/bin/pip install "setuptools<81"

In [5]:
# Cell 5 — Free up disk (miniforge package cache)
!rm -rf /kaggle/working/miniforge/pkgs
!df -h /kaggle/working

In [6]:
# Cell 6 — HuggingFace login
from huggingface_hub import login
login()  # paste your HF token

In [7]:
# Cell 7 — Download SD v1.5 (only the files we need)
!/kaggle/working/miniforge/envs/FreeFine/bin/huggingface-cli download \
    stable-diffusion-v1-5/stable-diffusion-v1-5 \
    --local-dir /kaggle/working/FreeFine/checkpoints/sd-15 \
    --local-dir-use-symlinks False \
    --include "model_index.json" \
              "scheduler/*" "tokenizer/*" "feature_extractor/*" \
              "text_encoder/config.json" "text_encoder/pytorch_model.bin" \
              "safety_checker/config.json" "safety_checker/pytorch_model.bin" \
              "unet/config.json" "unet/diffusion_pytorch_model.bin" \
              "vae/config.json" "vae/diffusion_pytorch_model.bin"
!df -h /kaggle/working

In [8]:
%%writefile /kaggle/working/FreeFine/run_demo.py
# Cell 8 — Write the demo script

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys, numpy as np, torch, cv2, random
from PIL import Image
sys.path.append('/kaggle/working/FreeFine')
os.chdir('/kaggle/working/FreeFine')

from src.demo.model import FreeFinePipeline
from src.utils.attention import register_attention_control, Attention_Modulator, register_attention_control_4bggen
from src.utils.vis_utils import (get_constrain_areas, re_edit_2d, dilate_mask,
                                  read_and_resize_mask_from_pil, read_and_resize_img_from_pil)
from diffusers import DDIMScheduler
from datasets import load_dataset

device = torch.device('cuda:0')
pretrained_model_path = "/kaggle/working/FreeFine/checkpoints/sd-15"
model = FreeFinePipeline.from_pretrained(pretrained_model_path, torch_dtype=torch.float16).to(device)
model.scheduler = DDIMScheduler.from_config(model.scheduler.config)

dataset_2d = load_dataset("CIawevy/GeoBench", "2d")['data']
id = 4865
sample = dataset_2d[id]
edit_param = sample['edit_param']
ori_img = read_and_resize_img_from_pil(sample['ori_img'])
ori_mask = read_and_resize_mask_from_pil(sample['ori_mask'])
obj_label = sample['obj_label']

constrain_areas = get_constrain_areas(mask_list=[ori_mask], ori_mask=ori_mask)
dil_ori_mask = dilate_mask(ori_mask, 20)
dil_ori_mask = np.where(constrain_areas, 0, dil_ori_mask)

Image.fromarray(ori_img).save('/kaggle/working/out_01_source.png')

controller = Attention_Modulator()
model.controller = controller
register_attention_control_4bggen(model, controller)
model.modify_unet_forward()
model.enable_attention_slicing()
model.enable_xformers_memory_efficient_attention()
seed_r = random.randint(0, 10**16)
generated_results = model.FreeFine_background_generation(
    ori_img, dil_ori_mask, 'empty scene',
    guidance_scale=3.5, eta=1.0, end_step=50, num_step=50, end_scale=0.5,
    start_step=1, share_attn=True, method_type='tca',
    local_text_edit=True, local_perturbation=True, verbose=True,
    seed=seed_r, return_intermediates=False, latent_blended=False)

mask_blurred = cv2.GaussianBlur(dil_ori_mask, (1, 1), 0) / 255
mask_np = 1 - (1 - dil_ori_mask) * (1 - mask_blurred)
inp_back_ground = (ori_img * (1 - mask_np) + generated_results * mask_np).astype(generated_results.dtype)
Image.fromarray(inp_back_ground).save('/kaggle/working/out_03_bg.png')

def parse_edit_param(p):
    dx, dy, dz, rx, ry, rz, sx, sy, sz = p
    return [dx, dy, rz, sx, sy]
parsed = parse_edit_param(edit_param)
coarse_input, target_mask, _ = re_edit_2d(ori_img, ori_mask, parsed, inp_back_ground)
Image.fromarray(coarse_input).save('/kaggle/working/out_04_coarse.png')

controller = Attention_Modulator(start_layer=10)
model.controller = controller
register_attention_control(model, controller)
model.modify_unet_forward()
model.enable_attention_slicing()
model.enable_xformers_memory_efficient_attention()
seed_r = random.randint(0, 10**16)
final = model.FreeFine_generation(
    ori_img=ori_img, ori_mask=ori_mask, coarse_input=coarse_input,
    target_mask=target_mask, guidance_text=obj_label,
    guidance_scale=7.5, eta=1.0, end_scale=0.0, end_step=50,
    num_step=50, start_step=35, seed=seed_r, draw_mask=None,
    return_intermediates=False, use_auto_draw=True,
    reduce_inp_artifacts=True, cons_area=target_mask)
Image.fromarray(final).save('/kaggle/working/out_05_final.png')
print("DONE")

In [9]:
# Cell 9 — Run the demo
!/kaggle/working/miniforge/envs/FreeFine/bin/python /kaggle/working/FreeFine/run_demo.py

In [10]:
# Cell 10 — Display results
from IPython.display import Image as IPImage, display
for f in ['out_01_source.png', 'out_03_bg.png', 'out_04_coarse.png', 'out_05_final.png']:
    print(f); display(IPImage(f'/kaggle/working/{f}'))

Cell A — Inspect GeoBenchMeta (run once, then delete)

In [11]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # 5–10× faster downloads

# First, just peek at config names without downloading data:
from huggingface_hub import HfApi
api = HfApi()
info = api.dataset_info("CIawevy/GeoBenchMeta")
print(info.card_data)  # shows config names

In [12]:
from datasets import get_dataset_config_names
print(get_dataset_config_names("CIawevy/GeoBenchMeta"))

In [13]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("CIawevy/GeoBenchMeta", repo_type="dataset")
# Print just the top-level structure
top = sorted(set(f.split('/')[0] for f in files))
print("Top-level:", top)
# And a sample of paths
for f in files[:30]: print(f)

In [14]:
from huggingface_hub import hf_hub_download
ann   = hf_hub_download("CIawevy/GeoBenchMeta", "annotation_sc.json", repo_type="dataset")
loader = hf_hub_download("CIawevy/GeoBenchMeta", "load_geobench.py",  repo_type="dataset")
print(ann); print(loader)

In [15]:
!head -100 {loader}
import json
with open(ann) as f: data = json.load(f)
print(type(data), len(data) if hasattr(data,'__len__') else '')
# Print one entry to see fields:
first = data[0] if isinstance(data, list) else next(iter(data.values()))
print(first if not isinstance(first, dict) else list(first.keys()))
print(first)

CELL A

In [16]:
from huggingface_hub import snapshot_download
import json
from pathlib import Path

ROOT = snapshot_download(
    "CIawevy/GeoBenchMeta", repo_type="dataset",
    allow_patterns=["Geo-Bench-SC/**", "annotation_sc.json"],
    local_dir="/kaggle/working/geobench_sc")
ROOT = Path(ROOT)
print("Downloaded to:", ROOT)
!du -sh /kaggle/working/geobench_sc

with open(ROOT / "annotation_sc.json") as f:
    raw = json.load(f)

samples = []
for scene_id, scene in raw.items():
    for obj_id, obj in scene["instances"].items():
        for edit_id, e in obj.items():
            samples.append({
                "id": f"{scene_id}_{obj_id}_{edit_id}",
                "caption": scene["4v_caption"],
                **e,
            })
print(f"Total SC cases: {len(samples)}")
print("First sample keys:", list(samples[0].keys()))

Cell A — Φόρτωση και προετοιμασία δεδομένων
Κατέβασες μόνο το structure-completion μέρος του GeoBenchMeta dataset (όχι ολόκληρο το dataset των 18k αρχείων). Συγκεκριμένα:

Με snapshot_download και το allow_patterns=["Geo-Bench-SC/**", "annotation_sc.json"] τράβηξες τοπικά τα 142MB που χρειάζεσαι: τις εικόνες SC και το JSON με τα annotations.
Άνοιξες το annotation_sc.json που έχει φωλιασμένη δομή: scene → object → edit. Δηλαδή κάθε σκηνή έχει πολλά objects, κάθε object έχει πολλά edits.
Έκανες flatten αυτή τη δομή σε μια απλή λίστα 121 samples, όπου κάθε sample έχει ένα μοναδικό id (π.χ. "0_0_0" = scene 0, object 0, edit 0) και όλα τα paths που χρειάζεσαι (ori_img_path, coarse_input_path, tgt_mask_path, draw_mask κλπ).

Το αποτέλεσμα: 121 cases έτοιμα για επεξεργασία, ακριβώς όσα λέει το paper για το structure-completion subset.

Cell B — Experiment script

In [48]:
%%writefile /kaggle/working/FreeFine/run_experiment.py
import os, sys, json, argparse, gc, traceback
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import numpy as np, torch, cv2
from pathlib import Path
from PIL import Image
sys.path.append('/kaggle/working/FreeFine')
os.chdir('/kaggle/working/FreeFine')

from src.demo.model import FreeFinePipeline
from src.utils.attention import register_attention_control, Attention_Modulator
from src.utils.vis_utils import (read_and_resize_mask_from_pil,
                                 read_and_resize_img_from_pil)
from diffusers import DDIMScheduler

SEED, NUM_STEP = 0, 50
DEVICE   = torch.device('cuda:0')
DATA_ROOT = Path("/kaggle/working/geobench_sc")
OUT_ROOT  = Path('/kaggle/working/outputs')

# --- automatic mask methods ----------------------------------------------
def mask_geometry(ori_mask, target_mask, **_):
    a = (ori_mask > 0).astype(np.uint8)
    b = (target_mask > 0).astype(np.uint8)
    return ((a | b) & ~(a & b)).astype(np.uint8) * 255

def mask_boundary(target_mask, k=15, **_):
    m = (target_mask > 0).astype(np.uint8) * 255
    kern = np.ones((k, k), np.uint8)
    return cv2.dilate(m, kern) - cv2.erode(m, kern)

def mask_artifact(ori_img, coarse_input, target_mask, thresh=25, **_):
    a = ori_img if ori_img.ndim == 3 else np.stack([ori_img]*3, -1)
    b = coarse_input if coarse_input.ndim == 3 else np.stack([coarse_input]*3, -1)
    tm = target_mask if target_mask.ndim == 2 else target_mask[..., 0]
    diff = np.abs(a.astype(np.int16) - b.astype(np.int16)).mean(-1)
    region = cv2.dilate((tm > 0).astype(np.uint8) * 255,
                        np.ones((20, 20), np.uint8))
    return ((diff > thresh).astype(np.uint8) * 255) & region
    
MASK_FNS = {'geometry': mask_geometry,
            'boundary': mask_boundary,
            'artifact': mask_artifact}

def build_model():
    m = FreeFinePipeline.from_pretrained(
        "/kaggle/working/FreeFine/checkpoints/sd-15",
        torch_dtype=torch.float16).to(DEVICE)
    m.scheduler = DDIMScheduler.from_config(m.scheduler.config)
    m.enable_attention_slicing()
    m.enable_xformers_memory_efficient_attention()
    return m

def load_samples():
    with open(DATA_ROOT / "annotation_sc.json") as f:
        raw = json.load(f)
    out = []
    for sid, sc in raw.items():
        for oid, ob in sc["instances"].items():
            for eid, e in ob.items():
                out.append({"id": f"{sid}_{oid}_{eid}", **e})
    return out

def load_img(rel):  return read_and_resize_img_from_pil (Image.open(DATA_ROOT / rel).convert('RGB'))
def load_mask(rel): return read_and_resize_mask_from_pil(Image.open(DATA_ROOT / rel).convert('L'))

def run_refinement(model, ori_img, ori_mask, coarse, target_mask,
                   obj_label, draw_mask, use_auto):
    ctrl = Attention_Modulator(start_layer=10); model.controller = ctrl
    register_attention_control(model, ctrl); model.modify_unet_forward()
    return model.FreeFine_generation(
        ori_img=ori_img, ori_mask=ori_mask, coarse_input=coarse,
        target_mask=target_mask, guidance_text=obj_label,
        guidance_scale=7.5, eta=1.0, end_scale=0.0, end_step=NUM_STEP,
        num_step=NUM_STEP, start_step=35, seed=SEED, draw_mask=draw_mask,
        return_intermediates=False, use_auto_draw=use_auto,
        reduce_inp_artifacts=True, cons_area=target_mask)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--method', required=True,
        choices=['manual','auto_builtin','geometry','boundary','artifact'])
    ap.add_argument('--limit', type=int, default=None)
    args = ap.parse_args()

    out_dir = OUT_ROOT / args.method; out_dir.mkdir(parents=True, exist_ok=True)
    samples = load_samples()
    if args.limit: samples = samples[:args.limit]

    model = build_model()
    results = []
    for i, s in enumerate(samples):
        sid = s['id']
        try:
            ori_img     = load_img (s['ori_img_path'])
            ori_mask    = load_mask(s['ori_mask_path'])
            coarse      = load_img (s['coarse_input_path'])
            target_mask = load_mask(s['tgt_mask_path'])
            obj_label   = s['obj_label']

            if args.method == 'manual':
                dm = load_mask(s['draw_mask']); use_auto = False
            elif args.method == 'auto_builtin':
                dm, use_auto = None, True
            else:
                dm = MASK_FNS[args.method](
                    ori_mask=ori_mask, target_mask=target_mask,
                    ori_img=ori_img, coarse_input=coarse)
                use_auto = False

            final = run_refinement(model, ori_img, ori_mask, coarse,
                                   target_mask, obj_label, dm, use_auto)
            Image.fromarray(final).save(out_dir / f'{sid}.png')
            if dm is not None:
                Image.fromarray(dm).save(out_dir / f'{sid}_mask.png')
            results.append({'id': sid, 'status': 'ok'})
            print(f'[{i+1}/{len(samples)}] {sid} ok', flush=True)
        except Exception as e:
            traceback.print_exc()
            results.append({'id': sid, 'status': 'error', 'error': str(e)})
            print(f'[{i+1}/{len(samples)}] {sid} ERROR: {e}', flush=True)
        torch.cuda.empty_cache(); gc.collect()

    with open(out_dir / 'results.json', 'w') as f:
        json.dump(results, f, indent=2)

if __name__ == '__main__':
    main()

Cell B — Το πειραματικό script
Έγραψες το run_experiment.py που τρέχει το FreeFine refinement με διαφορετικά είδη completion mask. Τα βασικά κομμάτια:
Πέντε μέθοδοι masks (το --method argument):

manual — χρησιμοποιεί το χειροκίνητο draw_mask που ήρθε με το dataset (το baseline σου).
auto_builtin — βάζει use_auto_draw=True, αφήνει το FreeFine να φτιάξει μόνο του τη μάσκα.
geometry — symmetric difference μεταξύ source και target mask (όπου άλλαξε γεωμετρικά το object).
boundary — δαχτυλίδι γύρω από το target mask (dilate − erode).
artifact — ανιχνεύει που η coarse εικόνα διαφέρει πολύ από το original, εκεί υπάρχουν artifacts να διορθωθούν.

Σημαντική απλοποίηση σε σχέση με το αρχικό σου notebook: Το dataset σου δίνει έτοιμα το coarse_input και το target_mask. Δεν χρειάζεται να τρέξεις πια background generation ούτε re_edit_2d. Πας κατευθείαν στο refinement step — γι' αυτό κάθε run είναι ~3× πιο γρήγορο.
Σταθερότητα: SEED=0 παντού, ίδιες παράμετροι σε όλες τις μεθόδους — μόνο το draw_mask αλλάζει. Αυτό κάνει τη σύγκριση έγκυρη.
Output: Για κάθε μέθοδο, σώζει 121 PNG αποτελέσματα + ένα results.json με success/error logs σε ξεχωριστό φάκελο (/kaggle/working/outputs/<method>/).

Cell C — Smoke test

In [18]:
!/kaggle/working/miniforge/envs/FreeFine/bin/python \
    /kaggle/working/FreeFine/run_experiment.py --method manual --limit 3

Τρέχει το script με --method manual --limit 3. Δηλαδή: μόνο 3 samples, μόνο με τη manual μέθοδο, για να επιβεβαιώσεις ότι:

Το script ξεκινάει χωρίς crash (φόρτωση μοντέλου, paths, dataset).
Το refinement παράγει εικόνες (όχι None ή σκουπίδια).
Τα paths στο annotation_sc.json είναι σωστά και τα PNG υπάρχουν εκεί που νομίζουμε.

Γιατί το κάνουμε αυτό: Αν κάτι έχει σπάσει, το μαθαίνεις σε 30 δευτερόλεπτα αντί να περιμένεις 12 λεπτά για να σκάσει στο sample #45. Είναι το πιο σημαντικό βήμα πριν τρέξεις το πλήρες πείραμα του Cell D που διαρκεί ~1 ώρα συνολικά για τις 5 μεθόδους × 121 samples.

Cell D - run all 5 methods

Cell D1 — manual (το baseline, ~20 min)

In [26]:
!/kaggle/working/miniforge/envs/FreeFine/bin/python /kaggle/working/FreeFine/run_experiment.py --method manual

Cell D2 — auto_builtin

In [27]:
!/kaggle/working/miniforge/envs/FreeFine/bin/python /kaggle/working/FreeFine/run_experiment.py --method auto_builtin

Cell D3 — geometry

In [38]:
!/kaggle/working/miniforge/envs/FreeFine/bin/python /kaggle/working/FreeFine/run_experiment.py --method geometry

Cell D4 — boundary

In [39]:
!/kaggle/working/miniforge/envs/FreeFine/bin/python /kaggle/working/FreeFine/run_experiment.py --method boundary

Cell D5 — artifact

In [54]:
!/kaggle/working/miniforge/envs/FreeFine/bin/python /kaggle/working/FreeFine/run_experiment.py --method artifact

Cell D6 — sanity check

In [60]:
!for M in manual auto_builtin geometry boundary artifact; do echo -n "$M: "; ls /kaggle/working/outputs/$M/*.png 2>/dev/null | grep -v _mask | wc -l; done

Cell E — Evaluation pipeline

Cell E1 — Setup του metric environment (~10 min, μία φορά μόνο)

In [61]:
%%bash
CONDA=/kaggle/working/miniforge/bin/conda
$CONDA create -n metric python=3.11.11 -y
PIP=/kaggle/working/miniforge/envs/metric/bin/pip
$PIP install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
$PIP install -r /kaggle/working/FreeFine/evaluation/metrics/requirements.txt
cd /kaggle/working/FreeFine/evaluation/metrics/
[ -f bpe_simple_vocab_16e6.txt.gz ] || wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz
df -h /kaggle/working

In [62]:
%%bash
CONDA=/kaggle/working/miniforge/bin/conda
$CONDA install -n metric pip -y
ls /kaggle/working/miniforge/envs/metric/bin/pip

Cell E2 — Φτιάξε ένα eval JSON ανά μέθοδο

In [63]:
import json
from pathlib import Path

DATA_ROOT = Path("/kaggle/working/geobench_sc")
OUT_ROOT  = Path("/kaggle/working/outputs")
EVAL_DIR  = Path("/kaggle/working/eval_jsons"); EVAL_DIR.mkdir(exist_ok=True)
METHODS   = ['manual', 'auto_builtin', 'geometry', 'boundary', 'artifact']
PATH_KEYS = ['ori_img_path', 'coarse_input_path', 'ori_mask_path', 'tgt_mask_path']

with open(DATA_ROOT / "annotation_sc.json") as f:
    raw = json.load(f)

for method in METHODS:
    out, n = {}, 0
    for sid, sc in raw.items():
        new_sc = {'4v_caption': sc['4v_caption'], 'instances': {}}
        for oid, ob in sc['instances'].items():
            new_ob = {}
            for eid, e in ob.items():
                gen = OUT_ROOT / method / f'{sid}_{oid}_{eid}.png'
                if not gen.exists():
                    continue
                e_abs = dict(e)
                for k in PATH_KEYS:
                    if k in e_abs:
                        e_abs[k] = str(DATA_ROOT / e_abs[k])
                e_abs['gen_img_path'] = str(gen)
                new_ob[eid] = e_abs; n += 1
            if new_ob: new_sc['instances'][oid] = new_ob
        if new_sc['instances']: out[sid] = new_sc

    with open(EVAL_DIR / f'eval_{method}.json', 'w') as f:
        json.dump(out, f, indent=2)
    print(f'{method}: {n} cases')

Cell E3 — Τρέξε τον evaluator για όλες τις μεθόδους

In [67]:
%%bash
PIP=/kaggle/working/miniforge/envs/metric/bin/pip
$PIP show image-reward 2>&1 | head -3
$PIP show hpsv2 2>&1 | head -3
$PIP show clip 2>&1 | head -3
cat /kaggle/working/FreeFine/evaluation/metrics/requirements.txt

In [71]:
%%bash
PIP=/kaggle/working/miniforge/envs/metric/bin/pip
$PIP install "setuptools<60" wheel
$PIP install --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
$PIP show clip 2>&1 | grep -E "^Name|^Version"

In [72]:
!/kaggle/working/miniforge/envs/metric/bin/python -c "import clip; print('OK', clip.__file__)" 2>&1

In [76]:
!/kaggle/working/miniforge/envs/metric/bin/pip install "transformers==4.48.3"

In [77]:
%%writefile /kaggle/working/run_eval.py
import sys, json, argparse, os
sys.path.insert(0, '/kaggle/working/FreeFine/evaluation/metrics')
os.chdir('/kaggle/working/FreeFine/evaluation/metrics')

# IRS optional
try:
    from image_reward import calculate_irs
    HAS_IRS = True
except Exception as e:
    print(f'[skip IRS] {type(e).__name__}: {e}', file=sys.stderr)
    HAS_IRS = False

from human_preference_score import calculate_hps
from VBench.background_consistency import calculate_bgc
from VBench.subject_consistency import calculate_subc
from wrap_error import calculate_we

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--path', required=True)
    ap.add_argument('--gen_img_key', default='gen_img_path')
    args = ap.parse_args()

    with open(args.path) as f:
        data = json.load(f)

    result = {}
    if HAS_IRS:
        result['IRS'] = calculate_irs(data, args.gen_img_key)
    result['HPS']    = calculate_hps (data, args.gen_img_key)
    result['BGC']    = calculate_bgc (data, args.gen_img_key)
    result['SUBC']   = calculate_subc(data, args.gen_img_key)
    result['WRAP_E'] = calculate_we  (data, args.gen_img_key)

    print('-----Result-----')
    for k, v in result.items():
        print(f'{k}: {v}')

if __name__ == '__main__':
    main()

In [79]:
%%bash
SRC=/kaggle/working/FreeFine/evaluation/metrics/bpe_simple_vocab_16e6.txt.gz
DST=/kaggle/working/miniforge/envs/metric/lib/python3.11/site-packages/hpsv2/src/open_clip/bpe_simple_vocab_16e6.txt.gz

if [ ! -f "$SRC" ]; then
  echo "Downloading bpe vocab..."
  wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -O "$SRC"
fi

cp "$SRC" "$DST"
ls -la "$DST"

In [80]:
%%bash
SRC=/kaggle/working/FreeFine/evaluation/metrics/bpe_simple_vocab_16e6.txt.gz
CLIP_DIR=/kaggle/working/miniforge/envs/metric/lib/python3.11/site-packages/clip
[ -d "$CLIP_DIR" ] && cp "$SRC" "$CLIP_DIR/" && ls -la "$CLIP_DIR/bpe_simple_vocab_16e6.txt.gz"

In [83]:
%%bash
/kaggle/working/miniforge/envs/metric/bin/pip install hf_transfer
PY=/kaggle/working/miniforge/envs/metric/bin/python
$PY /kaggle/working/run_eval.py --path /kaggle/working/eval_jsons/eval_manual.json 2>&1 | tee /kaggle/working/eval_logs/manual.log

In [84]:
%%bash
PY=/kaggle/working/miniforge/envs/metric/bin/python
for M in auto_builtin geometry boundary artifact; do
  echo "===== $M ====="
  $PY /kaggle/working/run_eval.py \
    --path /kaggle/working/eval_jsons/eval_${M}.json \
    2>&1 | tee /kaggle/working/eval_logs/${M}.log
done

Cell E4 — Συγκέντρωσε τα results σε πίνακα

In [85]:
import re, pandas as pd
from pathlib import Path

LOGS = Path('/kaggle/working/eval_logs')
rows = []
for method in ['manual', 'auto_builtin', 'geometry', 'boundary', 'artifact']:
    log = (LOGS / f'{method}.log').read_text()
    m = re.search(r'-----Result-----(.*)', log, re.S)
    if not m:
        print(f'No result section in {method}.log'); continue
    metrics = {}
    for line in m.group(1).strip().splitlines():
        if ':' in line:
            k, v = line.split(':', 1)
            try: metrics[k.strip()] = float(v.strip())
            except ValueError: pass
    rows.append({'method': method, **metrics})

df = pd.DataFrame(rows).set_index('method')
print(df.round(4))
df.to_csv('/kaggle/working/results_summary.csv')
print("\nSaved to /kaggle/working/results_summary.csv")

In [86]:
%%bash
cd /kaggle/working
zip -r thesis_results.zip \
  outputs/ \
  eval_jsons/ \
  eval_logs/ \
  results_summary.csv \
  geobench_sc/annotation_sc.json \
  FreeFine/run_experiment.py \
  run_eval.py
ls -lh thesis_results.zip